In [1]:
if (!require("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", quiet = TRUE, verbose = FALSE)
}

if (!require("stringr", quietly = TRUE)) {
  install.packages("stringr", quiet = TRUE, verbose = FALSE)
}

if (!require("data.table", quietly = TRUE)) {
  install.packages("data.table", quiet = TRUE, verbose = FALSE)
}

suppressPackageStartupMessages({
    library(tidyverse)
    library(stringr)
    library(data.table)
})

-- Attaching packages ------------------------------------------------------------------------------- tidyverse 1.3.2 --
v ggplot2 3.4.1      v purrr   1.0.0 
v tibble  3.2.0      v dplyr   1.0.10
v tidyr   1.2.1      v stringr 1.5.0 
v readr   2.1.3      v forcats 0.5.2 
-- Conflicts ---------------------------------------------------------------------------------- tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()

Attaching package: 'data.table'


The following objects are masked from 'package:dplyr':

    between, first, last


The following object is masked from 'package:purrr':

    transpose




#### Function to clean marker names

In [2]:
clean_marker <- function(marker) {
  marker <- tolower(marker)
  marker <- gsub("gsa-|seq-", "", marker)
  marker <- dplyr::recode(marker, "ilmnseq_6:35378798" = "rs9658134")
  return(marker)
}

#### Loading Data
* variants used as features for GBDT
* SNP-based GWAS logistic regression result

In [19]:
df <- fread("../feature_tsv/binary_feature_pruned_subject_removed.tsv")
colnames_df <- data.frame(variants = colnames(df)[1:260]) %>% 
  separate(., col = variants, into = c("gene", "MARKER"), sep = "_", remove=FALSE) %>% 
  select(MARKER)

snp <- fread("../admixture/variant_population/mtx_case_control_logistic_regression_result_09122023.tsv") %>% 
  select(-c(SNP,P) ) %>% 
  mutate(MARKER = clean_marker(MARKER))

setdiff(colnames_df$MARKER, snp$MARKER)

[1] "rs7761870"   "rs6457821"   "rs112012380" "rs3823433"   "rs6457813"  
[6] "rs9658134"

* The logistic regression data has MAF thus, filtered out low MAF. __--maf 0.01__, --geno 0.05, --hwe 0.000001 midp include-nonctrl, --mind 0.05  __GH__(exclude MAF < 0.01) :Methotrexate_QC_complete (ds102210)


* The initial feature table contains variant that has __noMAF__ --geno 0.05, --hwe 0.000001 midp include-nonctrl, --mind 0.05, __No Genotype Harmonizer__ CPNDS_8.03_variant&sampleQC_complete_noMAF_noGH (ds102004)

Cindy's speculation for the discrepancy:

__Data Inconsistency__: If the case and control groups are from different platforms or sources, the SNP may be harmonized out if it doesn't appear consistently across all datasets? 

In [4]:
snp <- snp %>%
inner_join(colnames_df) %>%
rename(OR = ORX, marker = MARKER, allele = A1)

Joining, by = "MARKER"


##### Save table

In [5]:
write.table(snp, "../admixture/variant_population/snp_frequency_variant.tsv", row.names = FALSE)

#### Reload ensembl ancestry_allele_frequency table (python API) 

In [6]:
ancestry_allele <- 
fread("../admixture/variant_population/ensembl_ancestry_allele_frequecy.tsv") %>% 
  distinct() %>% 
  mutate(population = str_replace(population, "1000GENOMES:phase_3:", "")) %>% 
  filter(population %in% c("AMR", "AFR", "EAS","EUR", "SAS") )

table(ancestry_allele$population)


AFR AMR EAS EUR SAS 
498 507 458 508 499 

In [7]:
ancestry_af <- snp %>% 
  left_join(ancestry_allele, by=c("marker", "allele")) %>% 
  pivot_wider(., names_from = population, values_from = frequency) %>%
  mutate(`effect allele`= case_when(
    OR > 1 ~ "increase risk",
    OR < 1 ~ "decrease risk",
    OR == 1 ~ "no effect" ))

#### Attach variants to  WNT or IL6 and the 6 genes PRKCD,PIK3R2, AGT, LRP5, CSNK1A1, PPARD: "5580, 5296, 183, 4041, 1452, 5467"


In [8]:
jagsetAnnot <- fread("../jag.set.annot.annot")
# covert all rsid to lower case
jagsetAnnot$V1 <- tolower(jagsetAnnot$V1)
# remove GSA- from colnames 
jagsetAnnot$V1 <- gsub("gsa-|seq-", "", jagsetAnnot$V1)
#change ilmnseq term
jagsetAnnot$V1[jagsetAnnot$V1 =="ilmnseq_6:35378798"] <- "rs9658134" 

jagsetAnnot <- jagsetAnnot %>%
  filter(V3 == "WNT" |V3 == "IL6") %>% 
  filter(V2 %in% c("5580", "5296", "183", "4041", "1452", "5467") ) %>% 
  mutate(symbols = case_when(
    grepl("5580", V2) ~ "PRKCD",
    grepl("5296", V2) ~ "PIK3R2",
    grepl("183", V2) ~ "AGT",
    grepl("4041", V2) ~ "LRP5",
    grepl("1452", V2) ~ "CSNK1A1",
    grepl("5467", V2) ~ "PPARD"
  ))

colnames(jagsetAnnot) <- c("marker", "entrezId", "pathways", "gene_symbols")

In [9]:
head(ancestry_af, 2)

marker,allele,OR,AFR,AMR,EAS,EUR,SAS,effect allele
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
rs853463,T,1.199,0.42208775,0.4697406,0.4394841,0.4850895,0.3251534,increase risk
rs853464,A,1.177,0.07791225,0.1080692,0.1984127,0.1312127,0.2423313,increase risk


In [10]:
# merge dataframes by rsid

ancestry_af <- left_join(ancestry_af, jagsetAnnot, by = "marker") 

In [11]:
colnames(ancestry_af)

[1] "marker"        "allele"        "OR"            "AFR"          
 [5] "AMR"           "EAS"           "EUR"           "SAS"          
 [9] "effect allele" "entrezId"      "pathways"      "gene_symbols"

In [27]:
population_cols = ancestry_af %>% 
select(c('AFR','AMR','EAS','EUR','SAS'))

present_df <- ancestry_af %>% 
mutate(max_col = apply(population_cols, 1, function(x) names(population_cols)[which.max(x)])) %>% 
                       select(gene_symbols, pathways, marker, allele, `effect allele`, `AFR`, `AMR`, `EAS`, `EUR`, `SAS`) 

In [28]:
write.csv(present_df , "../feature_tsv/ancestry_allele_frequency/ancestry_allele_frequency.tsv", quote=FALSE, row.names=FALSE)